# PCCP断丝信号特征挖掘

本 notebook 按开发方案组织完整流程：读取六类已计算特征向量，做质量检查、单特征判别、冗余分析、mRMR、Bootstrap 稳定性、跨流速一致性和最终特征分级。核心实现位于 `src/pccp_feature_mining`，notebook 只负责配置、运行和展示结果。

## 1. 环境与输入配置

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    """向上查找.git，保证从notebooks目录启动时也能定位项目根目录。"""
    p = start.resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / '.git').exists():
            return candidate
    return p

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pccp_feature_mining.config import FeatureInput, MiningConfig
from pccp_feature_mining.data_loader import load_feature_dataset
from pccp_feature_mining.quality_control import build_dataset_summary, build_feature_quality, feature_list_frame
from pccp_feature_mining.feature_discrimination import evaluate_feature_discrimination
from pccp_feature_mining.feature_selection import build_relevance_series, run_mrmr_ranking, build_final_ranking
from pccp_feature_mining.feature_redundancy import compute_correlation_matrix, high_correlation_pairs, build_correlation_clusters
from pccp_feature_mining.bootstrap_stability import run_bootstrap_stability
from pccp_feature_mining.cross_flow_analysis import evaluate_cross_flow_features
from pccp_feature_mining.report_generator import write_csv, write_markdown_summary
from pccp_feature_mining.visualization import plot_final_ranking, plot_top_feature_boxplots

print('项目根目录:', PROJECT_ROOT)

In [ ]:
FEATURE_INPUTS = [
    FeatureInput('FL00', PROJECT_ROOT / r'outputs/DATA09_v0-flow_features_20260904_121425/DATA09_v0-flow_features'),
    FeatureInput('FL05', PROJECT_ROOT / r'outputs/DATA09_v0-flow_features_20260904_121425/DATA09_v0.5-flow_features'),
    FeatureInput('BK00', PROJECT_ROOT / r'outputs/DATA09_multi_label_single_event_features/v0-bk_features_BK00'),
    FeatureInput('QJ00', PROJECT_ROOT / r'outputs/DATA09_multi_label_single_event_features/v0-qj_features_QJ00'),
    FeatureInput('BK05', PROJECT_ROOT / r'outputs/DATA09_multi_label_single_event_features/v05-bk_features_BK05'),
    FeatureInput('QJ05', PROJECT_ROOT / r'outputs/DATA09_multi_label_single_event_features/v05-qj_features_QJ05'),
]

CONFIG = MiningConfig(
    output_dir=PROJECT_ROOT / 'outputs/PCCP_feature_mining_notebook',
    feature_inputs=tuple(FEATURE_INPUTS),
    min_feature_csv_columns=100,
    random_state=42,
    correlation_threshold=0.92,
    bootstrap_rounds=200,
    bootstrap_top_ks=(10, 20, 30),
    mrmr_top_n=80,
    max_correlation_rows=6000,
    max_rows_per_label=None,  # 快速调试可设为1000；正式分析保持None。
)

RUN_DIR = CONFIG.output_dir / 'manual_run'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR

## 2. 数据读取与质量检查

In [ ]:
dataset = load_feature_dataset(
    feature_inputs=CONFIG.feature_inputs,
    min_feature_csv_columns=CONFIG.min_feature_csv_columns,
    max_rows_per_label=CONFIG.max_rows_per_label,
    random_state=CONFIG.random_state,
)
df = dataset.frame
feature_cols = list(dataset.feature_columns)

dataset_summary = build_dataset_summary(df)
feature_quality = build_feature_quality(df, feature_cols)
feature_list = feature_list_frame(feature_cols)

write_csv(dataset.load_report, RUN_DIR / 'load_report.csv')
write_csv(dataset_summary, RUN_DIR / 'dataset_summary.csv')
write_csv(feature_list, RUN_DIR / 'feature_list.csv')
write_csv(feature_quality, RUN_DIR / 'quality_report.csv')

print(f'样本行数: {len(df):,}, 特征数: {len(feature_cols):,}')
display(dataset_summary)
display(dataset.load_report[dataset.load_report['status'] != 'ok'])

## 3. 单特征判别能力

In [ ]:
feature_discrimination = evaluate_feature_discrimination(df, feature_cols, random_state=CONFIG.random_state)
write_csv(feature_discrimination, RUN_DIR / 'feature_discrimination.csv')

display(feature_discrimination[feature_discrimination['comparison'] == 'BK_NONBK'].head(30))
display(feature_discrimination[feature_discrimination['comparison'] == 'BK_QJ'].head(30))

## 4. 冗余分析与mRMR

In [ ]:
relevance = build_relevance_series(feature_discrimination, comparison='BK_NONBK')
corr = compute_correlation_matrix(
    df,
    feature_cols,
    method='spearman',
    max_rows=CONFIG.max_correlation_rows,
    random_state=CONFIG.random_state,
)
corr.to_csv(RUN_DIR / 'spearman_correlation_matrix.csv', encoding='utf-8-sig')

high_pairs = high_correlation_pairs(corr, threshold=CONFIG.correlation_threshold)
correlation_cluster = build_correlation_clusters(corr, relevance, threshold=CONFIG.correlation_threshold)
mrmr_rank = run_mrmr_ranking(
    relevance,
    corr,
    top_n=CONFIG.mrmr_top_n,
    redundancy_weight=CONFIG.mrmr_redundancy_weight,
)

write_csv(high_pairs, RUN_DIR / 'high_correlation_pairs.csv')
write_csv(correlation_cluster, RUN_DIR / 'correlation_cluster.csv')
write_csv(mrmr_rank, RUN_DIR / 'mrmr_rank.csv')

display(mrmr_rank.head(30))
display(high_pairs.head(30))

## 5. Bootstrap稳定性

In [ ]:
feature_stability = run_bootstrap_stability(
    df,
    feature_cols,
    rounds=CONFIG.bootstrap_rounds,
    top_ks=CONFIG.bootstrap_top_ks,
    random_state=CONFIG.random_state,
)
write_csv(feature_stability, RUN_DIR / 'feature_stability.csv')
display(feature_stability.head(30))

## 6. 跨流速一致性

In [ ]:
cross_flow_feature = evaluate_cross_flow_features(df, feature_cols)
write_csv(cross_flow_feature, RUN_DIR / 'cross_flow_feature.csv')
display(cross_flow_feature.head(30))

## 7. 最终特征排序与分级

In [ ]:
final_feature_ranking = build_final_ranking(
    feature_columns=feature_cols,
    discrimination=feature_discrimination,
    stability=feature_stability,
    cross_flow=cross_flow_feature,
    redundancy=correlation_cluster,
    mrmr_rank=mrmr_rank,
)
write_csv(final_feature_ranking, RUN_DIR / 'final_feature_ranking.csv')
write_markdown_summary(RUN_DIR / 'summary_report.md', dataset_summary, final_feature_ranking, dataset.load_report)

plot_final_ranking(final_feature_ranking, RUN_DIR / 'plots/final_feature_ranking_top30.png', top_n=30)
plot_top_feature_boxplots(
    df,
    final_feature_ranking.head(CONFIG.max_plot_features)['feature'].tolist(),
    RUN_DIR / 'plots/top_feature_boxplots.png',
    max_features=12,
)

display(final_feature_ranking.head(50))
print('输出目录:', RUN_DIR)

## 8. 一键运行入口

若不需要逐段查看中间结果，也可以直接运行下面的封装入口。该入口会自动创建带时间戳的输出目录，并生成方案要求的四个核心文件：`feature_discrimination.csv`、`feature_stability.csv`、`cross_flow_feature.csv`、`final_feature_ranking.csv`。

In [ ]:
from pccp_feature_mining.run_all import run_pccp_feature_mining

# summary = run_pccp_feature_mining(CONFIG)
# summary